# 🧠 How Do We Move and Process Data Efficiently? Memory, Batching & High-Dimensional Data

<a href="https://colab.research.google.com/github/rubenfonnegra/HPC/blob/master/Sem_08/Memory_Batching_HD_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/rubenfonnegra/HPC/tree/master/Sem_08/Memory_Batching_HD_Data.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a>


> **Performance depends not only on how fast we compute, but on how efficiently data reaches the computation.**

### Learning Goals
By the end of this notebook, students should be able to:
- Analyze how data representation affects performance, comparing numerical precision, memory footprint, and computational throughput.
- Evaluate the impact of memory access patterns and locality on execution time, cache efficiency, and data movement.
- Investigate the relationship between batch size, memory usage, and throughput in high-dimensional computational pipelines.
- Compare dense and sparse data representations and determine when sparse storage can provide meaningful memory or performance benefits.
- Measure the cost of CPU–GPU data transfers and distinguish between kernel execution time and end-to-end application performance.
- Identify when an optimization reduces memory consumption, execution time, or both, recognizing that lower memory usage does not necessarily imply faster computation.

---

> **📘 High Performance Computing (HPC)** \
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo \
> © 2026 · Educational use with attribution



## Setup and Baseline

We generate a realistic dense high-dimensional dataset with correlated/redundant dimensions using latent factors.


We will use one coherent pipeline throughout the notebook:

> **high-dimensional data → normalization → matrix projection → nonlinear transform → final score**

The goal is to measure how **precision, memory layout, batching, sparsity, CPU ↔ GPU movement, and prefetching** affect the same computational workload.


### Hypothesis
The vectorized whole-array baseline should compute efficiently, but normalization and projection will create substantial temporary memory traffic.

We establish runtime, throughput, input footprint, RAM availability, and GPU availability as the reference for later experiments.



In [ ]:
import os, gc, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from scipy import sparse
from concurrent.futures import ThreadPoolExecutor

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
N_SAMPLES, N_FEATURES, N_LATENT, N_PROJ = 50000, 1000, 32, 32
REPEATS = 3

def mib(n): return n / 1024**2
def array_mib(x): return mib(x.nbytes)
def rss_mib(): return mib(psutil.Process(os.getpid()).memory_info().rss)

def benchmark_function(
    func,
    repeat=3,
    warmup=1,
    is_gpu = False,
    **kwargs,
):
    """Return median wall-clock time for one pipeline function."""
    times = []

    for _ in range(warmup):
        _ = func(**kwargs)

    for _ in range(repeat):
        gc.collect()

        if is_gpu and CUDA_AVAILABLE:
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        _ = func(**kwargs)

        if is_gpu and CUDA_AVAILABLE:
            torch.cuda.synchronize()

        times.append(time.perf_counter() - t0)

    times = np.asarray(times)
    return {"times": times, "median_s": float(np.median(times)),
            "mean_s": float(times.mean()), "std_s": float(times.std())}


def generate_data(n=N_SAMPLES, p=N_FEATURES, k=N_LATENT, dtype=np.float32):
    r = np.random.default_rng(SEED)
    latent = r.normal(size=(n, k)).astype(dtype)
    loadings = r.normal(scale=.35, size=(k, p)).astype(dtype)
    X = latent @ loadings
    X += r.normal(scale=.50, size=X.shape).astype(dtype)
    return np.ascontiguousarray(X)

X = generate_data()
W = rng.normal(scale=1/np.sqrt(N_FEATURES), size=(N_FEATURES, N_PROJ)).astype(np.float32)

print("Shape:", X.shape, "| dtype:", X.dtype, "| elements:", f"{X.size:,}")
print("Dataset:", f"{array_mib(X):.1f} MiB", "| process RSS:", f"{rss_mib():.1f} MiB")
print("Available RAM:", f"{mib(psutil.virtual_memory().available):.1f} MiB")

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    print("CUDA available:", CUDA_AVAILABLE)
    if CUDA_AVAILABLE:
        p = torch.cuda.get_device_properties(0)
        print("GPU:", p.name, "| memory:", f"{mib(p.total_memory):.1f} MiB")
except Exception as e:
    torch, CUDA_AVAILABLE = None, False
    print("PyTorch/CUDA unavailable:", e)

In [ ]:
def baseline_pipeline(X, W):
    mean = X.mean(axis=0, dtype=X.dtype)
    std = X.std(axis=0, dtype=X.dtype) + np.asarray(1e-6, dtype=X.dtype)
    Z = (X - mean) / std
    H = np.tanh(Z @ W.astype(X.dtype, copy=False))
    return H.mean(axis=1)

b0 = benchmark_function(lambda: baseline_pipeline(X, W))
baseline_time = b0["median_s"]
baseline_output = baseline_pipeline(X, W)

baseline_df = pd.DataFrame([{
    "pipeline": "whole-array baseline",
    "dtype": str(X.dtype),
    "median_time_s": baseline_time,
    "std_s": b0["std_s"],
    "throughput_samples_s": N_SAMPLES / baseline_time,
    "input_memory_MiB": array_mib(X),
    "process_RSS_MiB": rss_mib()
}])

display(baseline_df.round(3))
print("Repeated times:", np.round(b0["times"], 4))

## Memory Representation and Precision

We compare `float64`, `float32`, and `float16`.

### Hypothesis
Lower precision reduces storage and bandwidth demand, but **memory savings do not guarantee proportional speedup**. CPU instruction support, conversions, and library kernels matter too.

For each dtype we measure memory, repeated runtime, throughput, and numerical difference from `float64`.

In [ ]:
precision_rows, precision_outputs = [], {}

for dtype in [np.float64, np.float32, np.float16]:
    name = np.dtype(dtype).name
    Xd, Wd = X.astype(dtype, copy=True), W.astype(dtype, copy=True)
    try:
        def run():
            mean = Xd.mean(axis=0, dtype=dtype)
            eps = np.asarray(1e-3 if dtype == np.float16 else 1e-6, dtype=dtype)
            std = Xd.std(axis=0, dtype=dtype) + eps
            return np.tanh(((Xd - mean) / std) @ Wd).mean(axis=1)
        b = benchmark_function(run)
        precision_outputs[name] = run().astype(np.float64)
        precision_rows.append({
            "dtype": name, "bytes_per_value": np.dtype(dtype).itemsize,
            "dataset_memory_MiB": array_mib(Xd), "median_time_s": b["median_s"],
            "throughput_samples_s": N_SAMPLES/b["median_s"], "status": "ok"
        })
    except Exception as e:
        precision_rows.append({
            "dtype": name, "bytes_per_value": np.dtype(dtype).itemsize,
            "dataset_memory_MiB": array_mib(Xd), "median_time_s": np.nan,
            "throughput_samples_s": np.nan, "status": type(e).__name__
        })
    del Xd, Wd
    gc.collect()

ref = precision_outputs.get("float64")
for row in precision_rows:
    y = precision_outputs.get(row["dtype"])
    row["mean_abs_diff_vs_float64"] = np.nan if ref is None or y is None else float(np.mean(np.abs(y-ref)))

precision_df = pd.DataFrame(precision_rows)
display(precision_df.round(6))

valid = precision_df.dropna(subset=["median_time_s"])
plt.figure(figsize=(7,4)); plt.bar(valid["dtype"], valid["dataset_memory_MiB"])
plt.ylabel("Dataset memory (MiB)"); plt.title("Precision vs memory footprint"); plt.show()

plt.figure(figsize=(7,4)); plt.bar(valid["dtype"], valid["median_time_s"])
plt.ylabel("Median execution time (s)"); plt.title("Precision vs execution time"); plt.show()

m64 = precision_df.loc[precision_df.dtype=="float64","dataset_memory_MiB"].iloc[0]
m32 = precision_df.loc[precision_df.dtype=="float32","dataset_memory_MiB"].iloc[0]

print(f"Memory saved float64 → float32: {100*(1-m32/m64):.1f}%")

## Memory Locality and Access Patterns

NumPy exposes physical layout through `array.flags` and `array.strides`.

### Hypothesis
Access that follows contiguous memory should use cache lines and bandwidth more effectively than strongly strided access.

> **Same arithmetic does not necessarily mean same performance.**

In [ ]:
A = X[:12_000, :1_000]
AT = A.T
print("A: C contiguous =", A.flags["C_CONTIGUOUS"], "| strides =", A.strides)
print("A.T: C contiguous =", AT.flags["C_CONTIGUOUS"],
      "| F contiguous =", AT.flags["F_CONTIGUOUS"], "| strides =", AT.strides)

def row_slices(A):
    total = 0.0
    for i in range(A.shape[0]): total += float(np.sum(A[i,:]))
    return total

def column_slices(A):
    total = 0.0
    for j in range(A.shape[1]): total += float(np.sum(A[:,j]))
    return total

def transposed_row_slices(AT):
    total = 0.0
    for i in range(AT.shape[0]): total += float(np.sum(AT[i,:]))
    return total

experiments = {
    "contiguous row slices": lambda: row_slices(A),
    "strided column slices": lambda: column_slices(A),
    "transposed-view rows": lambda: transposed_row_slices(AT)
}

rows = []
for name, fn in experiments.items():
    b = benchmark_function(fn, repeat=3)
    rows.append({"access_pattern": name, "median_time_s": b["median_s"]})

locality_df = pd.DataFrame(rows)
locality_df["relative_to_fastest"] = locality_df.median_time_s/locality_df.median_time_s.min()
display(locality_df.round(4))

plt.figure(figsize=(8,4)); plt.bar(locality_df.access_pattern, locality_df.median_time_s)
plt.ylabel("Median time (s)"); plt.xticks(rotation=15, ha="right")
plt.title("Memory locality and access pattern"); plt.tight_layout(); plt.show()

## Batching and Chunked Processing

Batching limits the active working set but adds loop/kernel overhead. We compute normalization statistics once so every batch performs the same mathematical transformation.

### Hypothesis
Tiny batches should lose throughput to overhead. Larger batches should improve throughput until cache/memory pressure becomes important. There is **no universal optimal batch size**.

In [ ]:
GLOBAL_MEAN = X.mean(axis=0, dtype=np.float32)
GLOBAL_STD = X.std(axis=0, dtype=np.float32) + np.float32(1e-6)

def process_in_batches(X, W, batch_size):
    out = np.empty(len(X), dtype=np.float32)
    for start in range(0, len(X), batch_size):
        stop = min(start+batch_size, len(X))
        xb = X[start:stop]
        h = np.tanh(((xb-GLOBAL_MEAN)/GLOBAL_STD) @ W)
        out[start:stop] = h.mean(axis=1)
    return out

def approx_temp_mib(bs):
    b = min(bs, len(X))
    return mib(b*N_FEATURES*4 + b*N_PROJ*4 + b*4)

batch_sizes = [128, 512, 2048, 8192, 32768, N_SAMPLES]
rows = []
for bs in batch_sizes:
    b = benchmark_function(lambda bs=bs: process_in_batches(X,W,bs))
    rows.append({"batch_size":bs, "median_time_s":b["median_s"],
                 "throughput_samples_s":N_SAMPLES/b["median_s"],
                 "approx_temp_working_set_MiB":approx_temp_mib(bs)})
batch_df = pd.DataFrame(rows)
display(batch_df.round(3))

plt.figure(figsize=(7,4)); plt.plot(batch_df.batch_size,batch_df.throughput_samples_s,marker="o")
plt.xscale("log");
plt.xlabel("Batch size"); plt.ylabel("Samples/s")
plt.title("Batch size vs throughput"); plt.show()

plt.figure(figsize=(7,4)); plt.plot(batch_df.batch_size,batch_df.approx_temp_working_set_MiB,marker="o")
plt.xscale("log");
plt.xlabel("Batch size"); plt.ylabel("Approx. temporary MiB")
plt.title("Batch size vs temporary working set"); plt.show()

## CPU–GPU Data Movement

If CUDA is available, PyTorch measures:
1. CPU → GPU transfer
2. GPU computation
3. GPU → CPU transfer
4. end-to-end time

We compare repeated per-batch transfers with minimizing transfers.

### Hypothesis
Repeated PCIe transfers can erase part of the GPU advantage. Kernel time alone can therefore be misleading.

In [ ]:
def sync():
    if CUDA_AVAILABLE: torch.cuda.synchronize()

def gpu_experiment(X,W,batch_size=8192):
    device = torch.device("cuda")
    Wc = torch.from_numpy(W); Mc = torch.from_numpy(GLOBAL_MEAN); Sc = torch.from_numpy(GLOBAL_STD)

    # Per batch transfer
    c2g=comp=g2c=0.0; ttotal=time.perf_counter()
    for start in range(0,len(X),batch_size):
        xc = torch.from_numpy(X[start:start+batch_size])
        t=time.perf_counter()
        xg,wg,mg,sg = xc.to(device),Wc.to(device),Mc.to(device),Sc.to(device); sync()
        c2g += time.perf_counter()-t
        t=time.perf_counter()
        yg = torch.tanh(((xg-mg)/sg)@wg).mean(dim=1); sync()
        comp += time.perf_counter()-t
        t=time.perf_counter()
        _ = yg.cpu(); sync()
        g2c += time.perf_counter()-t
    totalA=time.perf_counter()-ttotal

    # One transfer
    ttotal=time.perf_counter()
    t=time.perf_counter()
    xg,wg,mg,sg = torch.from_numpy(X).to(device),Wc.to(device),Mc.to(device),Sc.to(device); sync()
    c2gB=time.perf_counter()-t
    t=time.perf_counter()
    yg=torch.tanh(((xg-mg)/sg)@wg).mean(dim=1); sync()
    compB=time.perf_counter()-t
    t=time.perf_counter()
    _=yg.cpu(); sync()
    g2cB=time.perf_counter()-t
    totalB=time.perf_counter()-ttotal
    del xg,wg,mg,sg,yg; torch.cuda.empty_cache()

    return pd.DataFrame([
        {"strategy":"A: Per-batch transfers","CPU2GPU_s":c2g,"compute_s":comp,"GPU2CPU_s":g2c,"total_s":totalA},
        {"strategy":"B: Minimize transfers","CPU2GPU_s":c2gB,"compute_s":compB,"GPU2CPU_s":g2cB,"total_s":totalB}
    ])

if CUDA_AVAILABLE:
    _=torch.randn(256,256,device="cuda")@torch.randn(256,256,device="cuda"); sync()
    gpu_df=gpu_experiment(X,W)
    display(gpu_df.round(4))
    gpu_df.set_index("strategy")[["CPU2GPU_s","compute_s","GPU2CPU_s"]].plot(kind="bar",figsize=(8,4))
    plt.ylabel("Time (s)"); plt.title("Data movement vs GPU computation")
    plt.xticks(rotation=10,ha="right"); plt.tight_layout(); plt.show()
else:
    print("CUDA unavailable: GPU execution skipped gracefully.")
    print("With CUDA, this cell separates CPU2GPU, compute, GPU2CPU, and end-to-end time.")

## Overlapping Loading and Computation

We simulate I/O/preprocessing latency with `time.sleep()` and compare sequential loading with one-batch-ahead prefetching.

**Sequential:** Load N → Process N → Load N+1  
**Prefetched:** Process N while preparing N+1

### Hypothesis
Prefetching should reduce idle time when loading and compute have comparable costs.

In [ ]:
PREFETCH_ROWS, PREFETCH_BATCH, LOAD_DELAY = 20000, 2048, 0.008
Xp = X[:PREFETCH_ROWS]

def load_batch(X,start,stop):
    time.sleep(LOAD_DELAY)
    return X[start:stop].copy()

def compute_batch(xb):
    return np.tanh(((xb-GLOBAL_MEAN)/GLOBAL_STD)@W).mean(axis=1)

def sequential(X):
    out=[]
    for s in range(0,len(X),PREFETCH_BATCH):
        xb=load_batch(X,s,min(s+PREFETCH_BATCH,len(X)))
        out.append(compute_batch(xb))
    return np.concatenate(out)

def prefetched(X):
    starts=list(range(0,len(X),PREFETCH_BATCH)); out=[]
    with ThreadPoolExecutor(max_workers=1) as ex:
        s=starts[0]
        future=ex.submit(load_batch,X,s,min(s+PREFETCH_BATCH,len(X)))
        for i,s in enumerate(starts):
            xb=future.result()
            if i+1<len(starts):
                n=starts[i+1]
                future=ex.submit(load_batch,X,n,min(n+PREFETCH_BATCH,len(X)))
            out.append(compute_batch(xb))
    return np.concatenate(out)

bseq=benchmark_function(lambda:sequential(Xp),repeat=3,warmup=1)
bpre=benchmark_function(lambda:prefetched(Xp),repeat=3,warmup=1)

prefetch_df=pd.DataFrame([
    {"pipeline":"sequential","median_time_s":bseq["median_s"]},
    {"pipeline":"prefetched","median_time_s":bpre["median_s"]}
])
prefetch_df["throughput_samples_s"]=PREFETCH_ROWS/prefetch_df.median_time_s
prefetch_df["speedup_vs_sequential"]=bseq["median_s"]/prefetch_df.median_time_s
display(prefetch_df.round(3))

plt.figure(figsize=(7,4)); plt.bar(prefetch_df.pipeline,prefetch_df.median_time_s)
plt.ylabel("Median time (s)"); plt.title("Overlap loading with computation"); plt.show()

## Final Optimization Challenge

We combine the evidence into an intentionally inefficient CPU pipeline and an improved pipeline.

### Baseline
- `float64`
- tiny batches
- unnecessary copies
- sequential loading

### Improved
- `float32`
- contiguous data
- larger practical batches
- fewer copies
- prefetched loading

### Hypothesis
The improved data path should reduce overhead and bytes moved. The actual speedup must be **measured**, not assumed.

In [ ]:
FINAL_ROWS=30000
X32=np.ascontiguousarray(X[:FINAL_ROWS],dtype=np.float32)
X64=X32.astype(np.float64); W64=W.astype(np.float64)
M32=X32.mean(0); S32=X32.std(0)+1e-6
M64=X64.mean(0); S64=X64.std(0)+1e-6

def inefficient(X,W,batch=128,delay=.002):
    out=[]
    for s in range(0,len(X),batch):
        time.sleep(delay)
        xb=X[s:s+batch].copy()
        xb=np.array(xb,copy=True)
        out.append(np.tanh(((xb-M64)/S64)@W).mean(1))
    return np.concatenate(out)

def optimized(X,batch=8192,delay=.002):
    starts=list(range(0,len(X),batch)); out=[]
    def loader(s):
        time.sleep(delay)
        return X[s:min(s+batch,len(X))]
    with ThreadPoolExecutor(max_workers=1) as ex:
        f=ex.submit(loader,starts[0])
        for i,s in enumerate(starts):
            xb=f.result()
            if i+1<len(starts): f=ex.submit(loader,starts[i+1])
            out.append(np.tanh(((xb-M32)/S32)@W).mean(1))
    return np.concatenate(out)

bad =benchmark_function(lambda:inefficient(X64,W64),repeat=3,warmup=1)
good=benchmark_function(lambda:optimized(X32),repeat=3,warmup=1)
tb,tg=bad["median_s"],good["median_s"]

final_df=pd.DataFrame([
    {"pipeline":"inefficient baseline","dtype":"float64","median_time_s":tb,
     "throughput_samples_s":FINAL_ROWS/tb,"input_memory_MiB":array_mib(X64),"speedup":1.0},
    {"pipeline":"optimized","dtype":"float32","median_time_s":tg,
     "throughput_samples_s":FINAL_ROWS/tg,"input_memory_MiB":array_mib(X32),"speedup":tb/tg}
])
display(final_df.round(3))
print(f"speedup = baseline_time / optimized_time = {tb/tg:.2f}x")

plt.figure(figsize=(7,4)); plt.bar(final_df.pipeline,final_df.median_time_s)
plt.ylabel("Median time (s)"); plt.title("Final pipeline comparison"); plt.show()

plt.figure(figsize=(7,4)); plt.bar(final_df.pipeline,final_df.throughput_samples_s)
plt.ylabel("Samples/s"); plt.title("Final throughput comparison"); plt.show()

## Final evidence table

| Hypothesis | Measurement | Result | Interpretation |
|---|---|---|---|
| Lower precision reduces memory traffic | Section 2 | Inspect your output | Did runtime follow memory reduction? |
| Better locality improves access efficiency | Section 3 | Inspect your output | Relate timing to strides/cache behavior |
| Batch size has a useful operating region | Section 4 | Inspect your output | Balance overhead and working set |
| Sparse storage helps sufficiently sparse data | Section 5 | Inspect your output | Separate storage and compute benefits |
| Transfers can reduce GPU advantage | Section 6 | Inspect your output | Compare compute with end-to-end time |
| Prefetching can hide loading latency | Section 7 | Inspect your output | Identify overlap |
| Combined changes can improve the data path | Section 8 | Inspect your output | Attribute gains to evidence |

### Final reflection
1. Which optimization produced the largest performance improvement?
2. Which optimization primarily reduced memory rather than execution time?
3. Did any optimization make performance worse? Why?
4. When would GPU acceleration not be worthwhile?
5. What would change if the dataset were ten times larger?

### Workflow to retain

**Measure → Find Data Bottleneck → Reduce Movement → Batch → Improve Locality → Reduce Representation → Measure Again**

HPC optimization is empirical and iterative: the objective is not merely to show that an “optimized” version is faster, but to explain **why** the measured behavior occurred.

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Computación Aplicada (HPC) courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *How Do We Move and Process Data Efficiently? Memory, Batching & High-Dimensional Data: HPC Notebook*. Institución Universitaria Pascual Bravo.